# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muneebulhaq02/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/muneebulhaq02/flyrank-ml-internship"
REPO_DIR = "FlyRank-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())

Working dir: /content/FlyRank-Internship/FlyRank-Internship


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1

The paper reports that Average Position was the most important feature for predicting the Health Score using a Random Forest model. However, the paper also explains that Health Score is partly constructed from Average Position and Impressions.

## Methodology Question

Since Average Position is already part of the Health Score calculation, how much of the reported feature importance reflects genuine predictive ability versus simply reproducing the formula used to build the target? A comparison using an independent target could strengthen this conclusion.

## Finding 2

The paper reports a Logistic Regression model with 71% holdout accuracy for distinguishing growing and declining pages.

## Methodology Question

The paper reports holdout accuracy but provides limited discussion of grouped or time-aware validation. Because content from the same client may appear in both training and testing data, evaluating with grouped or time-based validation could provide a more realistic estimate of model performance in deployment.





In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
paper_findings = {
    "Finding 1":
        "Average Position is reported as the strongest feature for predicting Health Score.",
    "Methodology Question 1":
        "Is this importance partly caused because Average Position is already used inside the Health Score formula?",

    "Finding 2":
        "Logistic Regression achieved approximately 71% holdout accuracy.",
    "Methodology Question 2":
        "Would grouped or time-aware validation produce similar performance?"
}

for k,v in paper_findings.items():
    print(f"{k}:")
    print(v)
    print()

Finding 1:
Average Position is reported as the strongest feature for predicting Health Score.

Methodology Question 1:
Is this importance partly caused because Average Position is already used inside the Health Score formula?

Finding 2:
Logistic Regression achieved approximately 71% holdout accuracy.

Methodology Question 2:
Would grouped or time-aware validation produce similar performance?



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The Week-5 Random Forest model was first evaluated using a standard random train/test split. To better represent deployment conditions, the model was also evaluated using GroupKFold, where all pages belonging to the same client remain in the same fold. This reduces the possibility of similar client-specific content appearing in both training and testing. The comparison below shows how the validation strategy affects the measured model performance and provides a more realistic estimate of generalization.

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# -------------------------
# Same features as ML-08
# -------------------------

numeric_features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level"
]

X = pd.get_dummies(
    df[numeric_features + categorical_features],
    columns=categorical_features,
    drop_first=True
)

X = X.fillna(0)

# Same target as ML-08
y = (df["trend_direction"] == "down").astype(int)

# Groups for honest validation
groups = df["client_id"]

# -------------------------
# Random split
# -------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

random_accuracy = accuracy_score(
    y_test,
    rf.predict(X_test)
)

# -------------------------
# Grouped split
# -------------------------

gkf = GroupKFold(n_splits=5)

train_idx, test_idx = next(
    gkf.split(X, y, groups)
)

rf.fit(
    X.iloc[train_idx],
    y.iloc[train_idx]
)

group_accuracy = accuracy_score(
    y.iloc[test_idx],
    rf.predict(X.iloc[test_idx])
)

comparison = pd.DataFrame({
    "Validation": ["Random Split", "Grouped Split"],
    "Accuracy": [random_accuracy, group_accuracy]
})

print(comparison)

      Validation  Accuracy
0   Random Split  0.690500
1  Grouped Split  0.584475


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final feature set was reviewed for possible information leakage before model evaluation. Trend-related variables used to construct the prediction label were excluded from the feature matrix, along with identifier columns such as client_id and content_id. Only historical search and content performance measurements available before the prediction point were used as model inputs. Client identifiers were retained only for grouped validation and never used for prediction.

In [9]:
excluded = [
    "trend_direction",
    "trend_pct",
    "client_id",
    "content_id"
]

print("Leakage Audit")
print("-" * 30)

print("Excluded columns:")
for col in excluded:
    print("-", col)

print("\nFeatures used for modeling:")
for col in X.columns:
    print("-", col)

print("\nSummary:")
print("✓ No client identifiers used as features.")
print("✓ No future-window variables used.")
print("✓ No label-derived variables included in X.")
print("✓ trend_direction was used only to create the target variable.")

Leakage Audit
------------------------------
Excluded columns:
- trend_direction
- trend_pct
- client_id
- content_id

Features used for modeling:
- impressions_90d
- clicks_90d
- ctr
- avg_position
- content_age_days
- search_volume
- competition
- cpc
- word_count
- char_count
- engagement_rate
- scroll_rate
- ai_traffic_pct
- content_type_feedly article
- content_type_keyword article
- main_intent_informational
- main_intent_navigational
- main_intent_transactional
- competition_level_LOW
- competition_level_MEDIUM

Summary:
✓ No client identifiers used as features.
✓ No future-window variables used.
✓ No label-derived variables included in X.
✓ trend_direction was used only to create the target variable.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

The original project objective was expressed as predicting which pages should be refreshed. This statement has been revised to better reflect the scope of the analysis. The model identifies pages whose historical performance patterns are associated with declining content and produces a decision-support ranking for manual review. The results describe observed relationships in the available data and should not be interpreted as proof that refreshing a page will improve future search performance.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
claims = {
    "Original":
        "The model predicts which pages should be refreshed.",

    "Rewritten":
        "The model identifies pages associated with decline and provides a decision-support ranking for manual review."
}

for k,v in claims.items():
    print(k)
    print(v)
    print()

Original
The model predicts which pages should be refreshed.

Rewritten
The model identifies pages associated with decline and provides a decision-support ranking for manual review.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.